In [ ]:
PROMPT_DIAGNOSIS = '''
You will be given structured ECG diagnosis_type input variables. Fill these from the caller when you run:

- age: <integer>
- gender: "<Male|Female|Other>"
- diagnosis_type: JSON object where each key is a possible diagnosis and value is a probability (0–100). Example: {"Inferior myocardial infarction": 95.0, "Atrial fibrillation": 30.0}. If no diagnoses are present, it is an empty JSON {}.

Task:
Produce exactly a JSON *list* of TWO objects (no surrounding text, no explanation, nothing else). Each object MUST contain exactly these keys: "type", "question", "answer". Use English only. The two objects must appear in this order:

1) Diagnosis type closed-QA  
2) Diagnosis type open-QA  

---

RULES FOR USING PROBABILITIES
- For closed-QA:
  - If there is at least one diagnosis ≥60: choose the highest-probability diagnosis as the correct option.
  - Distractors must be clinically plausible but must not overlap with any actual diagnosis in the JSON (regardless of probability).
  - Distractors may include a "Normal" option (phrased as one of: "No abnormality detected", "Normal ECG", "Within normal limits", "No significant abnormality") only if it is not the correct answer.
  - If no diagnosis ≥60 and the JSON is empty (or only low-probability values), then the correct answer must be one of the “Normal” variants.
- For open-QA:
  - If there is at least one diagnosis ≥60: output only the highest-probability diagnosis.
  - If no diagnosis ≥60: output a “Normal” variant (e.g., "Normal ECG", "Within normal limits", "No abnormality detected").
- The closed-QA and open-QA questions must not be identical; they should be phrased differently even if based on the same input.
- If multiple waveform abnormalities have similar probabilities (close values or clinical ambiguity), prefer making the closed-QA and open-QA answers different (e.g., select one as closed-QA correct option and another as open-QA answer).

---

GENERAL FORMATTING RULES
- All output must be valid JSON. Do not output any text outside the JSON array.
- The "question" field must be a single string.
- Every question must reference the patient’s age and gender, but the wording should vary. Examples of acceptable variants:
  * "Based on the ECG signal of a 65-year-old male patient, ..."
  * "For the ECG of a 70-year-old female, ..."
  * "This ECG from a 55-year-old male patient indicates ..."
  * "Considering the ECG tracing of a 60-year-old patient (female), ..."
- For closed-QA:  
  The question text must include ONLY four options labeled `A: ...; B: ...; C: ...; D: ...` (separated by semicolons).
  Example: `"Based on the ECG signal of a 65-year-old male patient, Which diagnosis is most likely? A: X; B: Y; C: Z; D: W"`
- Closed-QA answers must be exactly the option label plus content, e.g., `"A: Atrial fibrillation"`.
- Don't keep relating the correct answers to one certain option label.
- For open-QA: output the correct diagnosis (or a Normal variant).
'''

In [ ]:
PROMPT_WAVEFORM = '''
You will be given structured ECG waveform input variables. Fill these from the caller when you run:

- age: <integer>
- gender: "<Male|Female|Other>"
- waveform: JSON object where each key is a waveform abnormality and value is a probability (0–100). Example: {"Q waves present": 95.0, "ST depression": 30.0}. If no waveform is present, it is an empty JSON {}.

Task:
Produce exactly a JSON *list* of TWO objects (no surrounding text, no explanation, nothing else). Each object MUST contain exactly these keys: "type", "question", "answer". Use English only. The two objects must appear in this order:

1) Waveform closed-QA  
2) Waveform open-QA  

---

RULES FOR USING PROBABILITIES
- For closed-QA:
  - If there is at least one waveform ≥60: choose the highest-probability waveform as the correct option.
  - Distractors must be clinically plausible but must not overlap with any actual waveform in the JSON.
  - Distractors may include a "Normal" variant (phrased as one of: "No abnormality detected", "Normal ECG", "Within normal limits", "No significant abnormality") only if it is not the correct answer.
  - If no waveform ≥60 and the JSON is empty (or only low-probability values), then the correct answer must be a “Normal” variant.
- For open-QA:
  - If there is at least one waveform ≥60: output only the highest-probability waveform.
  - If no waveform ≥60: output a “Normal” variant (e.g., "Normal ECG", "Within normal limits", "No abnormality detected").
- The closed-QA and open-QA questions must not be identical; they should be phrased differently even if based on the same input.
- If multiple waveform abnormalities have similar probabilities (close values or clinical ambiguity), prefer making the closed-QA and open-QA answers different (e.g., select one as closed-QA correct option and another as open-QA answer).

---

GENERAL FORMATTING RULES
- All output must be valid JSON. Do not output any text outside the JSON array.
- The "question" field must be a single string.
- Every question must reference the patient’s age and gender, but the wording should vary. Examples of acceptable variants:
  * "Based on the ECG signal of a 65-year-old male patient, ..."
  * "For the ECG of a 70-year-old female, ..."
  * "This ECG from a 55-year-old male patient indicates ..."
  * "Considering the ECG tracing of a 60-year-old patient (female), ..."
- For closed-QA:  
  The question text must include ONLY four options labeled `A: ...; B: ...; C: ...; D: ...` (separated by semicolons).
  Example: `"Based on the ECG signal of a 70-year-old female patient, Which waveform abnormality is most likely? A: X; B: Y; C: Z; D: W"`
- Closed-QA answers must be exactly the option label plus content, e.g., `"B: ST depression"`.
- Don't keep relating the correct answers to one certain option label.
- For open-QA: output the correct waveform abnormality (or a Normal variant).
'''

In [ ]:
PROMPT_RHYTHM = '''
You will be given structured ECG rhythm input variables. Fill these from the caller when you run:

- age: <integer>
- gender: "<Male|Female|Other>"
- rhythm: JSON object where each key is a rhythm characteristic and value is a probability (0–100). Example: {"High QRS voltage": 90.0, "Low voltage": 20.0}. Empty JSON {} if none.

Task:
Produce exactly a JSON *list* of TWO objects (no surrounding text, no explanation, nothing else). Each object MUST contain exactly these keys: "type", "question", "answer". Use English only. The two objects must appear in this order:

1) Rhythm closed-QA  
2) Rhythm open-QA  

---

RULES FOR USING PROBABILITIES
- For closed-QA:
  - If there is at least one rhythm ≥60: choose the highest-probability symptom as the correct option.
  - Distractors must be clinically plausible but must not overlap with any actual rhythm in the JSON.
  - Distractors may include a "Normal" variant (phrased as one of: "No abnormality detected", "Normal ECG", "Within normal limits", "No significant abnormality") only if it is not the correct answer.
  - If no rhythm ≥60 and the JSON is empty (or only low-probability values), then the correct answer must be a “Normal” variant.
- For open-QA:
  - If there is at least one rhythm ≥60: output only the highest-probability rhythm.
  - If no rhythm ≥60: output a “Normal” variant (e.g., "Normal ECG", "Within normal limits", "No abnormality detected").
- The closed-QA and open-QA questions must not be identical; they should be phrased differently even if based on the same input.
- If multiple rhythm abnormalities have similar probabilities (close values or clinical ambiguity), prefer making the closed-QA and open-QA answers different (e.g., select one as closed-QA correct option and another as open-QA answer).

---

GENERAL FORMATTING RULES
- All output must be valid JSON. Do not output any text outside the JSON array.
- The "question" field must be a single string.
- Every question must reference the patient’s age and gender, but the wording should vary. Examples of acceptable variants:
  * "Based on the ECG signal of a 65-year-old male patient, ..."
  * "For the ECG of a 70-year-old female, ..."
  * "This ECG from a 55-year-old male patient indicates ..."
  * "Considering the ECG tracing of a 60-year-old patient (female), ..."
- For closed-QA:  
  The question text must include ONLY four options labeled `A: ...; B: ...; C: ...; D: ...` (separated by semicolons).
  Example: `"Based on the ECG signal of a 70-year-old female patient, Which rhythm abnormality is most likely? A: X; B: Y; C: Z; D: W"`
- Closed-QA answers must be exactly the option label plus content, e.g., `"C: High QRS voltage"`.
- Don't keep relating the correct answers to one certain option label.
- For open-QA: output the correct rhythm abnormality (or a Normal variant).
'''

In [ ]:
PROMPT_REPORT_PREDICTION = '''
You will be given structured ECG input variables. Fill these from the caller when you run:

- age: <integer>
- gender: "<Male|Female|Other>"
- diagnosis_type: JSON object where each key is a possible diagnosis and value is a probability (0–100).
- waveform: JSON object where each key is a waveform abnormality and value is a probability (0–100).
- rhythm: JSON object where each key is a rhythm characteristic and value is a probability (0–100).
- n_seconds: <integer> (number of seconds to forecast)

Task:
Produce exactly a JSON *list* of TWO objects (no surrounding text, no explanation, nothing else). Each object MUST contain exactly these keys: "type", "question", "answer". Use English only. The two objects must appear in this order:

1) Report generation  
2) Signal forecasting  

---

REPORT RULES (item 1)
- The "question" might be similar to: "Based on the ECG signal of a [age]-year-old [gender] patient, Generate a structured clinical 12-lead ECG report."
- The "answer" must be a high-quality structured report that would score highly under professional evaluation criteria:
  - Cover diagnosis completeness (mention key diagnoses, suspected findings, likelihood/severity).
  - Ensure waveform accuracy (correct anatomical region and abnormality description).
  - Ensure rhythm accuracy (baseline rhythm, arrhythmias, conduction issues, pacing).
  - Logical structure: findings organized clearly, relevant patient info included, anonymization preserved.
  - Use standard terminology (SCP-ECG), avoid overconfident or vague language.
  - Certainty is expressed with phrases like "highly likely", "possible", "less likely", guided by the probabilities.
- If no abnormality ≥60 is present in all inputs, the report must explicitly state that the ECG is within normal limits, but phrasing may vary (e.g., "Normal ECG", "Within normal limits", "No significant abnormality detected").

---

PREDICTION RULES (item 2)
- The "question" might be similar to:  
  `"Based on the ECG signal of a [age]-year-old [gender] patient, Predict the waveform of the next [n_seconds] seconds."`
- The "answer" must be **blank** (empty string).  

---

GENERAL FORMATTING RULES
- All output must be valid JSON. Do not output any text outside the JSON array.
'''

In [10]:
PROMPTS = [PROMPT_DIAGNOSIS, PROMPT_WAVEFORM, PROMPT_RHYTHM, PROMPT_REPORT_PREDICTION]
FOLDERS = ['diagnosis', 'waveform', 'rhythm', 'report_prediction']

In [ ]:
def generate_user(row, n_seconds, idx):
    user_template = f'''
- age: {int(row.age)}
- gender: {row.sex}'''
    match idx:
        case 0:
            user_template += f'''
- diagnosis_type: {row.diagnostic}
'''

        case 1:
            user_template += f'''
- waveform: {row.form}
'''

        case 2:
            user_template += f'''
- rhythm: {row.rhythm}
'''

        case 3:
            user_template += f'''
- diagnosis_type: {row.diagnostic}
- waveform: {row.form}
- rhythm: {row.rhythm}
- n_seconds: {n_seconds}
'''

    return user_template

In [ ]:
import json

def parse_model_response(response_text: str):
    try:
        response_text = response_text.strip()
        
        if response_text.startswith("```"):
            response_text = response_text.strip("`")
            if response_text.lower().startswith("json"):
                response_text = response_text[4:].strip()
        
        parsed = json.loads(response_text)
        
        if isinstance(parsed, list) and all(isinstance(x, dict) for x in parsed):
            return parsed
        else:
            return []
    except Exception:
        return []

In [ ]:
from openai import OpenAI

def get_qa(user, prompt):    
    key = ''

    client = OpenAI(
        base_url="",
        api_key=key
    )

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": prompt},
            {"role": "user", "content": user}
        ]
    )
    answer = response.choices[0].message.content

    # print(f"[SYSTEM]\n{prompt}")
    # print(f"[USER]\n{user}")
    # print(f"[ASSISTANT]\n{answer}\n")

    return parse_model_response(answer)